In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
#import seaborn as sns
import sklearn
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [2]:
from sklearn.linear_model import Ridge, RidgeCV, Lasso, LinearRegression, BayesianRidge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.utils import resample

In [3]:
from vaccination_functions import *
from acs_data_functions import *

In [4]:
tx_mmr_vac_rates = pd.read_csv("../TX_MMR_vax_rate_24_25.csv")
tx_mmr_vac_rates.head()

,Facility Number,School Type,School District or Name,Facility Address,County,MMR Vaccination Rate,Age Group
0,9057829000,Public School,A+ ACADEMY,"8225 BRUTON RD, DALLAS, TX, .",Dallas,96.77,Kindergarten
1,9109901000,Public School,ABBOTT ISD,"P O BOX 226, ABBOTT, TX, .",Hill,95.24,Kindergarten
2,9095901000,Public School,ABERNATHY ISD,"505 7TH ST, ABERNATHY, TX, .",Hale,93.22,Kindergarten
3,9221901000,Public School,ABILENE ISD,"P O BOX 981, ABILENE, TX, .",Taylor,97.76,Kindergarten
4,9014901000,Public School,ACADEMY ISD,"704 E MAIN ST, LITTLE RIVER, TX, .",Bell,98.04,Kindergarten


In [5]:
counties_with_vac = tx_mmr_vac_rates['County'].unique()
#tx_mmr_vac_rates['County'].unique()

In [6]:
acs_data_path = "../acs_data/"

age_df = pd.read_csv(acs_data_path+"tx_county_agePop_24.csv")
age_var_df = pd.read_csv(acs_data_path+"ageVars_acs.csv")

race_df = pd.read_csv(acs_data_path+"tx_county_racePop_24.csv")
race_var_df = pd.read_csv(acs_data_path+"raceVars_acs.csv")

education_df = pd.read_csv(acs_data_path+"tx_county_education_24.csv")
education_var_df = pd.read_csv(acs_data_path+"educationVars_acs.csv")

#enrollment_df = pd.read_csv(acs_data_path+"tx_county_enrollment_24.csv")
#enrollment_var_df = pd.read_csv(acs_data_path+"enrollmentVars_acs.csv")

enrollment_ext_df = pd.read_csv(acs_data_path+"tx_county_enrollment_24_extended.csv")
enrollment_var_ext_df = pd.read_csv(acs_data_path+"enrollmentVars_acs_extended.csv")

poverty_df = pd.read_csv(acs_data_path+"tx_county_poverty_24.csv")
poverty_var_df = pd.read_csv(acs_data_path+"povertyVars_acs.csv")


In [7]:
enroll_concepts = ['B14001_', 'B14002_', 'B14003_', 'B14006_', 'B14007', 'B14007_', 'B14007A_', 'B14007B_', 'B14007C_',
                   'B14007D_', 'B14007E_', 'B14007F_', 'B14007G_', 'B14007H_', 'B14007I_',]
enroll_df_dict = {}
for c in enroll_concepts:
    enroll_sub_df = enrollment_ext_df[enrollment_ext_df['variable'].str.contains(c)]
    enroll_df_dict[c] = enroll_sub_df

In [8]:
enroll_totals_df = enroll_df_dict['B14001_']
enroll_gender_grade_df = enroll_df_dict['B14002_']
enroll_gender_type_age_df = enroll_df_dict['B14003_']
enroll_poverty_df = enroll_df_dict['B14006_']
enroll_detailed_grade_race_df = enroll_df_dict['B14007']

In [9]:
age_with_var_all, age_with_var_no_gender = create_predictor_datasets(age_df, age_var_df, counties_with_vac)
race_with_var_all, race_with_var_no_gender = create_predictor_datasets(race_df, race_var_df, counties_with_vac)
education_with_var_all, education_with_var_no_gender = create_predictor_datasets(education_df, education_var_df, counties_with_vac)
#enrollment_with_var_all, enrollment_with_var_no_gender = create_predictor_datasets(enrollment_ext_df, enrollment_var_ext_df, counties_with_vac)
poverty_with_var_all, poverty_with_var_no_gender = create_predictor_datasets(poverty_df, poverty_var_df, counties_with_vac)

#poverty_with_var_all.head()

In [23]:
enrollment_gender_grade_with_var_all, enrollment_gender_grade_with_var_no_gender = create_predictor_datasets(enroll_gender_grade_df, enrollment_var_ext_df, counties_with_vac)
enrollment_totals_with_var_all, enrollment_totals_with_var_no_gender = create_predictor_datasets(enroll_totals_df, enrollment_var_ext_df, counties_with_vac)
enrollment_gender_type_with_var_all, enrollment_gender_type_with_var_no_gender = create_predictor_datasets(enroll_gender_type_age_df, enrollment_var_ext_df, counties_with_vac)
enrollment_poverty_with_var_all, enrollment_poverty_with_var_no_gender = create_predictor_datasets(enroll_poverty_df, enrollment_var_ext_df, counties_with_vac)
enrollment_grade_race_with_var_all, enrollment_grade_race_with_var_no_gender = create_predictor_datasets(enroll_detailed_grade_race_df, enrollment_var_ext_df, counties_with_vac)

In [18]:
enrollment_totals_with_var_all.columns = enrollment_totals_with_var_all.columns.str.replace({'_School_Enrollment_by_Level_of_School_for_the_Population_': '_School_Enrollment_'}, regex=True)
enrollment_totals_with_var_all.columns

Index(['Estimate_Total_Enrolled_in_school_Enrolled_in_college,_undergraduate_School_Enrollment_3_Years_and_Over',
       'Estimate_Total_Enrolled_in_school_Enrolled_in_grade_1_grade_4_School_Enrollment_3_Years_and_Over',
       'Estimate_Total_Enrolled_in_school_Enrolled_in_grade_5_grade_8_School_Enrollment_3_Years_and_Over',
       'Estimate_Total_Enrolled_in_school_Enrolled_in_grade_9_grade_12_School_Enrollment_3_Years_and_Over',
       'Estimate_Total_Enrolled_in_school_Enrolled_in_kindergarten_School_Enrollment_3_Years_and_Over',
       'Estimate_Total_Enrolled_in_school_Enrolled_in_nursery_school,_preschool_School_Enrollment_3_Years_and_Over',
       'Estimate_Total_Enrolled_in_school_Graduate_or_professional_school_School_Enrollment_3_Years_and_Over',
       'Estimate_Total_Enrolled_in_school_School_Enrollment_3_Years_and_Over',
       'Estimate_Total_Not_enrolled_in_school_School_Enrollment_3_Years_and_Over',
       'Estimate_Total_School_Enrollment_3_Years_and_Over'],
      dty

In [16]:
enrollment_gender_grade_with_var_all.columns = enrollment_gender_grade_with_var_all.columns.str.replace({'_Sex_by_School_Enrollment_by_Level_of_School_by_Type_of_School_for_the_Population_': '_School_Enrollment_'}, regex=True)
enrollment_gender_grade_with_var_all.columns

Index(['Estimate_Total_Female_Enrolled_in_school_Enrolled_in_college_undergraduate_Private_school_School_Enrollment_3_Years_and_Over',
       'Estimate_Total_Female_Enrolled_in_school_Enrolled_in_college_undergraduate_Public_school_School_Enrollment_3_Years_and_Over',
       'Estimate_Total_Female_Enrolled_in_school_Enrolled_in_college_undergraduate_School_Enrollment_3_Years_and_Over',
       'Estimate_Total_Female_Enrolled_in_school_Enrolled_in_grade_1_grade_4_Private_school_School_Enrollment_3_Years_and_Over',
       'Estimate_Total_Female_Enrolled_in_school_Enrolled_in_grade_1_grade_4_Public_school_School_Enrollment_3_Years_and_Over',
       'Estimate_Total_Female_Enrolled_in_school_Enrolled_in_grade_1_grade_4_School_Enrollment_3_Years_and_Over',
       'Estimate_Total_Female_Enrolled_in_school_Enrolled_in_grade_5_grade_8_Private_school_School_Enrollment_3_Years_and_Over',
       'Estimate_Total_Female_Enrolled_in_school_Enrolled_in_grade_5_grade_8_Public_school_School_Enrollment_3_

In [25]:
enrollment_poverty_with_var_all.columns = enrollment_poverty_with_var_all.columns.str.replace({'_in_the_Past_12_Months_by_School_Enrollment_by_Level_of_School_for_the_Population_': '_School_Enrollment_'}, regex=True)
enrollment_poverty_with_var_all.columns

Index(['Estimate_Total_Income_in_the_past_12_months_at_or_above_the_poverty_level_Enrolled_in_school_Enrolled_in_college_undergraduate_Poverty_Status_School_Enrollment_3_Years_and_Over',
       'Estimate_Total_Income_in_the_past_12_months_at_or_above_the_poverty_level_Enrolled_in_school_Enrolled_in_grade_1_grade_4_Poverty_Status_School_Enrollment_3_Years_and_Over',
       'Estimate_Total_Income_in_the_past_12_months_at_or_above_the_poverty_level_Enrolled_in_school_Enrolled_in_grade_5_grade_8_Poverty_Status_School_Enrollment_3_Years_and_Over',
       'Estimate_Total_Income_in_the_past_12_months_at_or_above_the_poverty_level_Enrolled_in_school_Enrolled_in_grade_9_grade_12_Poverty_Status_School_Enrollment_3_Years_and_Over',
       'Estimate_Total_Income_in_the_past_12_months_at_or_above_the_poverty_level_Enrolled_in_school_Enrolled_in_graduate_or_professional_school_Poverty_Status_School_Enrollment_3_Years_and_Over',
       'Estimate_Total_Income_in_the_past_12_months_at_or_above_the_pove

In [25]:
cleanMap = {':!!': '_', '!!': '_', ' to ': '_', ' years': '', ' County, Texas': '', ':': '', ' ': '_'}
concepts = enrollment_var_ext_df['concept'].replace(cleanMap, regex=True).unique()
concepts

<StringArray>
[                                                            'School_Enrollment_by_Level_of_School_for_the_Population_3_Years_and_Over',
                                    'Sex_by_School_Enrollment_by_Level_of_School_by_Type_of_School_for_the_Population_3_Years_and_Over',
                                                'Sex_by_School_Enrollment_by_Type_of_School_by_Age_for_the_Population_3_Years_and_Over',
                     'Poverty_Status_in_the_Past_12_Months_by_School_Enrollment_by_Level_of_School_for_the_Population_3_Years_and_Over',
                                      'School_Enrollment_by_Detailed_Level_of_School_for_the_Population_3_Years_and_Over_(White_Alone)',
                  'School_Enrollment_by_Detailed_Level_of_School_for_the_Population_3_Years_and_Over_(Black_or_African_American_Alone)',
          'School_Enrollment_by_Detailed_Level_of_School_for_the_Population_3_Years_and_Over_(American_Indian_and_Alaska_Native_Alone)',
                           

In [16]:
concepts[1]

'Sex by School Enrollment by Level of School by Type of School for the Population 3 Years and Over'

In [30]:
#predictor_df_no_gender_percents_concept1 = predictor_df_no_gender_percents
#predictor_df_no_gender_percents.columns[predictor_df_no_gender_percents.columns.str.contatins(concepts[1])]
list = [col for col in predictor_df_no_gender_percents.columns if concepts[2] in col]
predictor_df_no_gender_percents[list].columns

Index(['Estimate_Total_Sex_by_School_Enrollment_by_Type_of_School_by_Age_for_the_Population_3_Years_and_Over'], dtype='str', name='variable')

In [ ]:
for co in concepts:
    concept_col = [col for col in predictor_df_no_gender_percents.columns if co in col]
    concept_sub_df = predictor_df_no_gender_percents[concept_col]
    for ca in predictor_df_no_gender_percents.drop(columns=['Estimate_Total']).columns:
            predictor_df_no_gender_percents[ca] = predictor_df_no_gender_percents[ca]/predictor_df_no_gender_percents['Estimate_Total']

In [10]:
predictor_df_no_gender_percents.columns

Index(['Estimate_Total_Enrolled_in_school_Enrolled_in_college,_undergraduate_School_Enrollment_by_Detailed_Level_of_School_for_the_Population_3_Years_and_Over',
       'Estimate_Total_Enrolled_in_school_Enrolled_in_college,_undergraduate_School_Enrollment_by_Detailed_Level_of_School_for_the_Population_3_Years_and_Over_(American_Indian_and_Alaska_Native_Alone)',
       'Estimate_Total_Enrolled_in_school_Enrolled_in_college,_undergraduate_School_Enrollment_by_Detailed_Level_of_School_for_the_Population_3_Years_and_Over_(Asian_Alone)',
       'Estimate_Total_Enrolled_in_school_Enrolled_in_college,_undergraduate_School_Enrollment_by_Detailed_Level_of_School_for_the_Population_3_Years_and_Over_(Black_or_African_American_Alone)',
       'Estimate_Total_Enrolled_in_school_Enrolled_in_college,_undergraduate_School_Enrollment_by_Detailed_Level_of_School_for_the_Population_3_Years_and_Over_(Hispanic_or_Latino)',
       'Estimate_Total_Enrolled_in_school_Enrolled_in_college,_undergraduate_School_

In [8]:
age_with_var_no_gender['Estimate_Total_Under_5'] = age_with_var_all['Estimate_Total_Male_Under_5'] + age_with_var_all['Estimate_Total_Female_Under_5']
age_with_var_no_gender['Estimate_Total_5_9'] = age_with_var_all['Estimate_Total_Male_5_9'] + age_with_var_all['Estimate_Total_Female_5_9']
age_with_var_no_gender['Estimate_Total_10_14'] = age_with_var_all['Estimate_Total_Male_10_14'] + age_with_var_all['Estimate_Total_Female_10_14']
age_with_var_no_gender['Estimate_Total_15_17'] = age_with_var_all['Estimate_Total_Male_15_17'] + age_with_var_all['Estimate_Total_Female_15_17']
age_with_var_no_gender['Estimate_Total_18_19'] = age_with_var_all['Estimate_Total_Male_18_and_19'] + age_with_var_all['Estimate_Total_Female_18_and_19']
age_with_var_no_gender['Estimate_Total_21'] = age_with_var_all['Estimate_Total_Male_21'] + age_with_var_all['Estimate_Total_Female_21']
age_with_var_no_gender['Estimate_Total_22_24'] = age_with_var_all['Estimate_Total_Male_22_24'] + age_with_var_all['Estimate_Total_Female_22_24']
age_with_var_no_gender['Estimate_Total_25_29'] = age_with_var_all['Estimate_Total_Male_25_29'] + age_with_var_all['Estimate_Total_Female_25_29']
age_with_var_no_gender['Estimate_Total_30_34'] = age_with_var_all['Estimate_Total_Male_30_34'] + age_with_var_all['Estimate_Total_Female_30_34']
age_with_var_no_gender['Estimate_Total_35_39'] = age_with_var_all['Estimate_Total_Male_35_39'] + age_with_var_all['Estimate_Total_Female_35_39']


In [9]:
age_with_var_no_gender['Estimate_Total_40_44'] = age_with_var_all['Estimate_Total_Male_40_44'] + age_with_var_all['Estimate_Total_Female_40_44']
age_with_var_no_gender['Estimate_Total_45_49'] = age_with_var_all['Estimate_Total_Male_45_49'] + age_with_var_all['Estimate_Total_Female_45_49']
age_with_var_no_gender['Estimate_Total_50_54'] = age_with_var_all['Estimate_Total_Male_50_54'] + age_with_var_all['Estimate_Total_Female_50_54']
age_with_var_no_gender['Estimate_Total_55_59'] = age_with_var_all['Estimate_Total_Male_55_59'] + age_with_var_all['Estimate_Total_Female_55_59']
age_with_var_no_gender['Estimate_Total_60_61'] = age_with_var_all['Estimate_Total_Male_60_and_61'] + age_with_var_all['Estimate_Total_Female_60_and_61']
age_with_var_no_gender['Estimate_Total_62_64'] = age_with_var_all['Estimate_Total_Male_62_64'] + age_with_var_all['Estimate_Total_Female_62_64']
age_with_var_no_gender['Estimate_Total_65_66'] = age_with_var_all['Estimate_Total_Male_65_and_66'] + age_with_var_all['Estimate_Total_Female_65_and_66']
age_with_var_no_gender['Estimate_Total_67_69'] = age_with_var_all['Estimate_Total_Male_67_69'] + age_with_var_all['Estimate_Total_Female_67_69']
age_with_var_no_gender['Estimate_Total_70_74'] = age_with_var_all['Estimate_Total_Male_70_74'] + age_with_var_all['Estimate_Total_Female_70_74']
age_with_var_no_gender['Estimate_Total_75_79'] = age_with_var_all['Estimate_Total_Male_75_79'] + age_with_var_all['Estimate_Total_Female_75_79']
age_with_var_no_gender['Estimate_Total_80_84'] = age_with_var_all['Estimate_Total_Male_80_84'] + age_with_var_all['Estimate_Total_Female_80_84']
age_with_var_no_gender['Estimate_Total_85_over'] = age_with_var_all['Estimate_Total_Male_85_and_over'] + age_with_var_all['Estimate_Total_Female_85_and_over']

In [10]:
age_with_var_all_percent, age_with_var_no_gender_percent = create_predictor_percentages(age_with_var_all, age_with_var_no_gender)
race_with_var_all_percent, race_with_var_no_gender_percent = create_predictor_percentages(race_with_var_all, race_with_var_no_gender)
education_with_var_all_percent, education_with_var_no_gender_percent = create_predictor_percentages(education_with_var_all, education_with_var_no_gender)
enrollment_with_var_all_percent, enrollment_with_var_no_gender_percent = create_predictor_percentages(enrollment_with_var_all, enrollment_with_var_no_gender)

In [11]:
### Predictor datasets with vaccination rates for absolute values
race_vac_set_all, race_vac_set_no_gender = vac_predictor_merge(tx_mmr_vac_rates, race_with_var_all, race_with_var_no_gender)
age_vac_set_all, age_vac_set_no_gender = vac_predictor_merge(tx_mmr_vac_rates, age_with_var_all, age_with_var_no_gender)
education_vac_set_all, education_vac_set_no_gender = vac_predictor_merge(tx_mmr_vac_rates, education_with_var_all, education_with_var_no_gender)
enrollment_vac_set_all, enrollment_vac_set_no_gender = vac_predictor_merge(tx_mmr_vac_rates, enrollment_with_var_all, enrollment_with_var_no_gender)
poverty_vac_set_all, poverty_vac_set_no_gender = vac_predictor_merge(tx_mmr_vac_rates, poverty_with_var_all, poverty_with_var_no_gender)


#poverty_vac_set_all.head()

In [12]:
### Predictor datasets with vaccination rates for percentages
race_vac_set_all_percent, race_vac_set_no_gender_percent = vac_predictor_merge(tx_mmr_vac_rates, race_with_var_all_percent, race_with_var_no_gender_percent)
age_vac_set_all, age_vac_set_no_gender_percent = vac_predictor_merge(tx_mmr_vac_rates, age_with_var_all_percent, age_with_var_no_gender_percent)
education_vac_set_all_percent, education_vac_set_no_gender_percent = vac_predictor_merge(tx_mmr_vac_rates, education_with_var_all_percent, education_with_var_no_gender_percent)
enrollment_vac_set_all_percent, enrollment_vac_set_no_gender_percent = vac_predictor_merge(tx_mmr_vac_rates, enrollment_with_var_all_percent, enrollment_with_var_no_gender_percent)

c:\Users\19037\Documents\Measels_Vaccination\vacMeas\vaccination_functions.py:87: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  single_pred_df_all = pd.merge(predictor_df_all, vac_df_rel, on=['County'])


In [ ]:
"""
#### Education Test
y_edu = education_vac_set_no_gender['MMR Vaccination Rate']
x_edu = education_vac_set_no_gender.drop(columns=['MMR Vaccination Rate', 'County', 'School District or Name', 'Estimate_Total', 'B15003_025'])

###
eductaion_model = LinearRegression()
eductaion_model.fit(x_edu, y_edu)

coeff_edu = eductaion_model.coef_
intercept_edu = eductaion_model.intercept_

print("Education coefficients = ", coeff_edu)
print("Education intercept = ", intercept_edu)
"""

KeyError: "['School District or Name'] not found in axis"

In [ ]:
#education_results = pd.DataFrame({'Education Predictors': x_edu.columns, 'Coefficents': coeff_edu})
#education_results

,Education Predictors,Coefficents
0,Estimate_Total_10th_grade,0.001557
1,Estimate_Total_11th_grade,0.001213
2,"Estimate_Total_12th_grade,_no_diploma",0.002057
3,Estimate_Total_1st_grade,-0.009994
4,Estimate_Total_2nd_grade,0.007263
5,Estimate_Total_3rd_grade,0.005866
6,Estimate_Total_4th_grade,0.002936
7,Estimate_Total_5th_grade,-0.011827
8,Estimate_Total_6th_grade,0.001381
9,Estimate_Total_7th_grade,-0.001146


In [ ]:
#education_results['Correlation'] = ['Negative' if x < 0 else 'Positive' for x in education_results['Coefficents']]
#education_results

,Education Predictors,Coefficents,Correlation
0,Estimate_Total_10th_grade,0.001557,Positive
1,Estimate_Total_11th_grade,0.001213,Positive
2,"Estimate_Total_12th_grade,_no_diploma",0.002057,Positive
3,Estimate_Total_1st_grade,-0.009994,Negative
4,Estimate_Total_2nd_grade,0.007263,Positive
5,Estimate_Total_3rd_grade,0.005866,Positive
6,Estimate_Total_4th_grade,0.002936,Positive
7,Estimate_Total_5th_grade,-0.011827,Negative
8,Estimate_Total_6th_grade,0.001381,Positive
9,Estimate_Total_7th_grade,-0.001146,Negative


In [14]:
eudcation_intercept, education_results_df, education_fit_summary = linear_regression_fit(education_vac_set_no_gender.drop(columns=['B15003_025']))
eudcation_intercept_percent, education_results_df_percent, edu_percent_fit_summary = linear_regression_fit(education_vac_set_no_gender_percent.drop(columns=['B15003_025']))

race_intercept, race_results_df, race_fit_summary = linear_regression_fit(race_vac_set_no_gender)
race_intercept_percent, race_results_df_percent, race_percent_fit_summary = linear_regression_fit(race_vac_set_no_gender_percent)

enrollment_intercept, enrollment_results_df, enrollment_fit_summary = linear_regression_fit(enrollment_vac_set_no_gender)
enrollment_intercept_percent, enrollment_results_df_percent, enroll_percent_fit_summary = linear_regression_fit(enrollment_vac_set_no_gender_percent)

age_intercept, age_results_df, age_fit_summary= linear_regression_fit(age_vac_set_no_gender)
age_intercept_percent, age_results_df_percent, age_percent_fit_summary = linear_regression_fit(age_vac_set_no_gender_percent)

In [ ]:
#poverty_intercept, poverty_results_df = linear_regression_fit(poverty_vac_set_all, predictor_name='poverty_variables', sort_and_save=True)
poverty_intercept, poverty_results_df, poverty_fit_summary = linear_regression_fit(poverty_vac_set_all)
poverty_results_df

,Predictors,Coeff,Correlation,CI_Low,CI_High
0,Estimate_Total_Income_in_the_past_12_months_at...,-0.000258,Negative,-0.000345,-0.000170
1,Estimate_Total_Income_in_the_past_12_months_be...,0.000129,Positive,0.000084,0.000174


In [ ]:
results_array = [poverty_results_df, age_results_df, race_results_df, enrollment_results_df, education_results_df]

In [ ]:
poverty_results_df[poverty_results_df['Correlation'] ==  'Positive']['Predictors']

1    Estimate_Total_Income_in_the_past_12_months_be...
Name: Predictors, dtype: str

In [ ]:
age_results_df[age_results_df['Correlation'] ==  'Positive']['Predictors'].to_list()

['Estimate_Total_Under_5',
 'Estimate_Total_15_17',
 'Estimate_Total_22_24',
 'Estimate_Total_30_34',
 'Estimate_Total_35_39',
 'Estimate_Total_45_49',
 'Estimate_Total_65_66',
 'Estimate_Total_67_69',
 'Estimate_Total_70_74',
 'Estimate_Total_85_over']

In [ ]:
postive_predictors = []
for r in results_array:
    positive_list = r[r['Correlation'] ==  'Positive']['Predictors'].to_list()
    #postive_predictors.append(positive_list)
    for p in positive_list:
        postive_predictors.append(p)

In [ ]:
postive_predictors

['Estimate_Total_Income_in_the_past_12_months_below_the_poverty_level',
 'Estimate_Total_Under_5',
 'Estimate_Total_15_17',
 'Estimate_Total_22_24',
 'Estimate_Total_30_34',
 'Estimate_Total_35_39',
 'Estimate_Total_45_49',
 'Estimate_Total_65_66',
 'Estimate_Total_67_69',
 'Estimate_Total_70_74',
 'Estimate_Total_85_over',
 'Estimate_Total_Black_or_African_American_alone',
 'Estimate_Total_Two_or_More_Races',
 'Estimate_Total_Two_or_More_Races_Two_races_excluding_Some_Other_Race,_and_three_or_more_races',
 'Estimate_Total_Enrolled_in_school',
 'Estimate_Total_Enrolled_in_school_Enrolled_in_kindergarten',
 'Estimate_Total_Enrolled_in_school_Graduate_or_professional_school',
 'Estimate_Total_10th_grade',
 'Estimate_Total_11th_grade',
 'Estimate_Total_12th_grade,_no_diploma',
 'Estimate_Total_2nd_grade',
 'Estimate_Total_3rd_grade',
 'Estimate_Total_4th_grade',
 'Estimate_Total_6th_grade',
 'Estimate_Total_8th_grade',
 "Estimate_Total_Bachelor's_degree",
 "Estimate_Total_Master's_degree"

In [15]:
all_acs_variables_no_gender = combine_all_acs_variables(education_with_var_no_gender, race_with_var_no_gender, enrollment_with_var_no_gender, age_with_var_no_gender, poverty_with_var_all)
all_acs_variables_all = combine_all_acs_variables(education_with_var_all, race_with_var_all, enrollment_with_var_all, age_with_var_all, poverty_with_var_all)

In [ ]:
edu_race_df = pd.merge(education_with_var_no_gender.drop(columns=['Estimate_Total', 'B15003_025']), race_with_var_no_gender.drop(columns='Estimate_Total'), on=['County'])
enroll_age_df = pd.merge(enrollment_with_var_no_gender.drop(columns='Estimate_Total'), age_with_var_no_gender.drop(columns='Estimate_Total'), on=['County'])
all_acs_variables = pd.merge(pd.merge(edu_race_df, enroll_age_df, on=['County']), poverty_with_var_all, on=['County'])
all_acs_variables.head()

variable,Estimate_Total_10th_grade,Estimate_Total_11th_grade,"Estimate_Total_12th_grade,_no_diploma",Estimate_Total_1st_grade,Estimate_Total_2nd_grade,Estimate_Total_3rd_grade,Estimate_Total_4th_grade,Estimate_Total_5th_grade,Estimate_Total_6th_grade,Estimate_Total_7th_grade,...,Estimate_Total_60_61,Estimate_Total_62_64,Estimate_Total_65_66,Estimate_Total_67_69,Estimate_Total_70_74,Estimate_Total_75_79,Estimate_Total_80_84,Estimate_Total_85_over,Estimate_Total_Income_in_the_past_12_months_at_or_above_the_poverty_level_Did_not_work,Estimate_Total_Income_in_the_past_12_months_below_the_poverty_level
County,,,,,,,,,,,,,,,,,,,,,
Anderson,1235,1008,1386,15,51,32,57,81,202,231,...,1372,1746,1224,1802,2319,1643,1369,508,2137,2492
Andrews,308,204,363,41,28,0,46,18,27,281,...,621,661,248,305,640,193,276,144,447,822
Angelina,1172,1095,1267,0,12,5,80,163,154,626,...,2419,2871,1801,2540,4328,2675,1975,1403,3287,4337
Aransas,262,140,150,0,0,0,17,0,0,25,...,865,1222,871,1305,1898,1553,686,741,1468,1500
Archer,94,117,108,0,0,0,9,0,1,12,...,271,476,223,466,481,388,215,159,446,377


In [16]:
all_vac_set, all_vac_set_no_gender = vac_predictor_merge(tx_mmr_vac_rates, all_acs_variables_all, all_acs_variables_no_gender)
all_vac_set

,County,Estimate_Total_10th_grade,Estimate_Total_11th_grade,"Estimate_Total_12th_grade,_no_diploma",Estimate_Total_1st_grade,Estimate_Total_2nd_grade,Estimate_Total_3rd_grade,Estimate_Total_4th_grade,Estimate_Total_5th_grade,Estimate_Total_6th_grade,...,Estimate_Total_Male_65_and_66,Estimate_Total_Male_67_69,Estimate_Total_Male_70_74,Estimate_Total_Male_75_79,Estimate_Total_Male_80_84,Estimate_Total_Male_85_and_over,Estimate_Total_Male_Under_5,Estimate_Total_Income_in_the_past_12_months_at_or_above_the_poverty_level_Did_not_work,Estimate_Total_Income_in_the_past_12_months_below_the_poverty_level,MMR Vaccination Rate
0,Anderson,1235,1008,1386,15,51,32,57,81,202,...,639,962,980,707,590,168,1512,2137,2492,90.24
1,Anderson,1235,1008,1386,15,51,32,57,81,202,...,639,962,980,707,590,168,1512,2137,2492,93.00
2,Anderson,1235,1008,1386,15,51,32,57,81,202,...,639,962,980,707,590,168,1512,2137,2492,96.23
3,Anderson,1235,1008,1386,15,51,32,57,81,202,...,639,962,980,707,590,168,1512,2137,2492,93.75
4,Anderson,1235,1008,1386,15,51,32,57,81,202,...,639,962,980,707,590,168,1512,2137,2492,96.28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3077,Zapata,670,179,195,0,28,94,56,71,113,...,113,202,154,125,72,218,510,396,913,100.00
3078,Zavala,132,228,235,0,3,18,128,0,77,...,140,97,210,143,10,81,376,92,784,100.00
3079,Zavala,132,228,235,0,3,18,128,0,77,...,140,97,210,143,10,81,376,92,784,100.00
3080,Zavala,132,228,235,0,3,18,128,0,77,...,140,97,210,143,10,81,376,92,784,100.00


In [17]:
all_intercept, all_results_df, all_results_summary = linear_regression_fit(all_vac_set)
all_results_df

,Predictors,Coeff,Correlation,CI_Low,CI_High,Coeff Signficance
0,Estimate_Total_10th_grade,-0.004401,Negative,-0.014115,0.005312,Not Signficant
1,Estimate_Total_11th_grade,0.005688,Positive,-0.005161,0.016537,Not Signficant
2,"Estimate_Total_12th_grade,_no_diploma",-0.000552,Negative,-0.010218,0.009114,Not Signficant
3,Estimate_Total_1st_grade,-0.003069,Negative,-0.079894,0.073757,Not Signficant
4,Estimate_Total_2nd_grade,-0.019484,Negative,-0.071979,0.033010,Not Signficant
...,...,...,...,...,...,...
169,Estimate_Total_Male_80_84,-0.017441,Negative,-0.034214,-0.000669,Signficant
170,Estimate_Total_Male_85_and_over,0.004764,Positive,-0.013981,0.023509,Not Signficant
171,Estimate_Total_Male_Under_5,0.003509,Positive,-0.010838,0.017855,Not Signficant
172,Estimate_Total_Income_in_the_past_12_months_at...,-0.001516,Negative,-0.008692,0.005659,Not Signficant


In [18]:

all_acs_variables_percent = combine_all_acs_variables(education_with_var_all_percent, race_with_var_all_percent, enrollment_with_var_all_percent, age_with_var_all_percent, poverty_with_var_all, percent=True, gender=True)
all_acs_variables_no_gender_percent = combine_all_acs_variables(education_with_var_no_gender_percent, race_with_var_no_gender_percent, enrollment_with_var_no_gender_percent, age_with_var_no_gender_percent, poverty_with_var_all, percent=True)

c:\Users\19037\Documents\Measels_Vaccination\vacMeas\vaccination_functions.py:98: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  edu_race_df = pd.merge(edu_df.drop(columns=['Estimate_Total', 'B15003_025', 'Percent_Male', 'Percent_Female']), race_df.drop(columns=['Estimate_Total']), on=['County'])
c:\Users\19037\Documents\Measels_Vaccination\vacMeas\vaccination_functions.py:100: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  all_acs_variables = pd.merge(pd.merge(edu_race_df, enroll_age_df, on=['County']), poverty_df, on=['County'])

In [19]:
all_vac_set_percent, all_vac_set_no_gender_percent = vac_predictor_merge(tx_mmr_vac_rates, all_acs_variables_percent, all_acs_variables_no_gender_percent)
all_vac_set_no_gender_percent

c:\Users\19037\Documents\Measels_Vaccination\vacMeas\vaccination_functions.py:87: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  single_pred_df_all = pd.merge(predictor_df_all, vac_df_rel, on=['County'])


,County,Percent_10th_grade,Percent_11th_grade,"Percent_12th_grade,_no_diploma",Percent_1st_grade,Percent_2nd_grade,Percent_3rd_grade,Percent_4th_grade,Percent_5th_grade,Percent_6th_grade,...,Percent_62_64,Percent_65_66,Percent_67_69,Percent_70_74,Percent_75_79,Percent_80_84,Percent_85_over,Estimate_Total_Income_in_the_past_12_months_at_or_above_the_poverty_level_Did_not_work,Estimate_Total_Income_in_the_past_12_months_below_the_poverty_level,MMR Vaccination Rate
0,Anderson,0.026141,0.021336,0.029337,0.000318,0.001080,0.000677,0.001207,0.001715,0.004276,...,0.029877,0.020945,0.030836,0.039682,0.028115,0.023426,0.008693,2137,2492,90.24
1,Anderson,0.026141,0.021336,0.029337,0.000318,0.001080,0.000677,0.001207,0.001715,0.004276,...,0.029877,0.020945,0.030836,0.039682,0.028115,0.023426,0.008693,2137,2492,93.00
2,Anderson,0.026141,0.021336,0.029337,0.000318,0.001080,0.000677,0.001207,0.001715,0.004276,...,0.029877,0.020945,0.030836,0.039682,0.028115,0.023426,0.008693,2137,2492,96.23
3,Anderson,0.026141,0.021336,0.029337,0.000318,0.001080,0.000677,0.001207,0.001715,0.004276,...,0.029877,0.020945,0.030836,0.039682,0.028115,0.023426,0.008693,2137,2492,93.75
4,Anderson,0.026141,0.021336,0.029337,0.000318,0.001080,0.000677,0.001207,0.001715,0.004276,...,0.029877,0.020945,0.030836,0.039682,0.028115,0.023426,0.008693,2137,2492,96.28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3077,Zapata,0.070810,0.018918,0.020609,0.000000,0.002959,0.009934,0.005918,0.007504,0.011943,...,0.022758,0.017629,0.031573,0.027527,0.023120,0.016834,0.024203,396,913,100.00
3078,Zavala,0.019835,0.034260,0.035312,0.000000,0.000451,0.002705,0.019234,0.000000,0.011570,...,0.009881,0.015512,0.030705,0.053442,0.023268,0.007650,0.026349,92,784,100.00
3079,Zavala,0.019835,0.034260,0.035312,0.000000,0.000451,0.002705,0.019234,0.000000,0.011570,...,0.009881,0.015512,0.030705,0.053442,0.023268,0.007650,0.026349,92,784,100.00
3080,Zavala,0.019835,0.034260,0.035312,0.000000,0.000451,0.002705,0.019234,0.000000,0.011570,...,0.009881,0.015512,0.030705,0.053442,0.023268,0.007650,0.026349,92,784,100.00


In [20]:
all_intercept_percent, all_results_df_percent, all_perecent_summary = linear_regression_fit(all_vac_set_no_gender_percent)
all_results_df_percent

,Predictors,Coeff,Correlation,CI_Low,CI_High,Coeff Signficance
0,Percent_10th_grade,-51.876026,Negative,-207.448090,103.696037,Not Signficant
1,Percent_11th_grade,-25.266434,Negative,-194.126648,143.593780,Not Signficant
2,"Percent_12th_grade,_no_diploma",18.793564,Positive,-136.828351,174.415480,Not Signficant
3,Percent_1st_grade,402.856807,Positive,-271.517144,1077.230759,Not Signficant
4,Percent_2nd_grade,134.656530,Positive,-346.413840,615.726902,Not Signficant
...,...,...,...,...,...,...
61,Percent_75_79,41.489372,Positive,-417.149404,500.128167,Not Signficant
62,Percent_80_84,82.357470,Positive,-375.710004,540.424964,Not Signficant
63,Percent_85_over,140.640087,Positive,-323.894770,605.174964,Not Signficant
64,Estimate_Total_Income_in_the_past_12_months_at...,-0.000172,Negative,-0.000290,-0.000055,Signficant


In [ ]:
#all_intercept_percent

np.float64(32.0060896349136)

In [21]:
y = all_vac_set_no_gender_percent['MMR Vaccination Rate']
x = all_vac_set_no_gender_percent.drop(columns=['MMR Vaccination Rate', 'County']) 
x_with_constant = sm.add_constant(x)
model = sm.OLS(y, x_with_constant).fit()

In [ ]:
### 95% confidence intervals for all coeff
intervals = model.conf_int(alpha=0.05).drop('const').reset_index()
intervals

,index,0,1
0,Percent_10th_grade,-207.448090,103.696037
1,Percent_11th_grade,-194.126648,143.593780
2,"Percent_12th_grade,_no_diploma",-136.828351,174.415480
3,Percent_1st_grade,-271.517144,1077.230759
4,Percent_2nd_grade,-346.413840,615.726902
...,...,...,...
61,Percent_75_79,-417.149404,500.128167
62,Percent_80_84,-375.710004,540.424964
63,Percent_85_over,-323.894770,605.174964
64,Estimate_Total_Income_in_the_past_12_months_at...,-0.000290,-0.000055


In [ ]:
intervals.rename(columns={'index': 'Predictors', 0: 'CI_Low', 1: 'CI_High'}, inplace=True)

In [ ]:
intervals.columns

Index(['Predictors', 0, 1], dtype='object')

In [ ]:
all_results_df_percent.merge(intervals)

,Predictors,Coeff,Correlation,CI_Low,CI_High
0,Percent_10th_grade,-51.876026,Negative,-207.448090,103.696037
1,Percent_11th_grade,-25.266434,Negative,-194.126648,143.593780
2,"Percent_12th_grade,_no_diploma",18.793564,Positive,-136.828351,174.415480
3,Percent_1st_grade,402.856807,Positive,-271.517144,1077.230759
4,Percent_2nd_grade,134.656530,Positive,-346.413840,615.726902
...,...,...,...,...,...
61,Percent_75_79,41.489372,Positive,-417.149404,500.128167
62,Percent_80_84,82.357470,Positive,-375.710004,540.424964
63,Percent_85_over,140.640087,Positive,-323.894770,605.174964
64,Estimate_Total_Income_in_the_past_12_months_at...,-0.000172,Negative,-0.000290,-0.000055


In [ ]:
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                             OLS Regression Results                             
================================================================================
Dep. Variable:     MMR Vaccination Rate   R-squared:                       0.069
Model:                              OLS   Adj. R-squared:                  0.049
Method:                   Least Squares   F-statistic:                     3.586
Date:                  Mon, 18 May 2026   Prob (F-statistic):           5.19e-19
Time:                          14:04:53   Log-Likelihood:                -11341.
No. Observations:                  3082   AIC:                         2.281e+04
Df Residuals:                      3019   BIC:                         2.319e+04
Df Model:                            62                                         
Covariance Type:              nonrobust                                         
==========================================================================================================================================================
                                                                                             coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------------------------------------------------
const                                                                                     19.0135     43.760      0.434      0.664     -66.790     104.817
Percent_10th_grade                                                                       -51.8760     79.343     -0.654      0.513    -207.448     103.696
Percent_11th_grade                                                                       -25.2664     86.120     -0.293      0.769    -194.127     143.594
Percent_12th_grade,_no_diploma                                                            18.7936     79.369      0.237      0.813    -136.828     174.415
Percent_1st_grade                                                                        402.8568    343.937      1.171      0.242    -271.517    1077.231
Percent_2nd_grade                                                                        134.6565    245.350      0.549      0.583    -346.414     615.727
Percent_3rd_grade                                                                        104.9821    125.106      0.839      0.401    -140.319     350.283
Percent_4th_grade                                                                        -50.4822    109.972     -0.459      0.646    -266.110     165.145
Percent_5th_grade                                                                       -110.7415    143.525     -0.772      0.440    -392.158     170.675
Percent_6th_grade                                                                        -66.7424    100.194     -0.666      0.505    -263.198     129.713
Percent_7th_grade                                                                         39.5144     80.857      0.489      0.625    -119.027     198.056
Percent_8th_grade                                                                       -130.3520    112.821     -1.155      0.248    -351.567      90.863
Percent_9th_grade                                                                        -86.5460     91.967     -0.941      0.347    -266.871      93.779
Percent_Associate's_degree                                                               -12.6137     77.456     -0.163      0.871    -164.486     139.259
Percent_Bachelor's_degree                                                                 -2.3659     75.031     -0.032      0.975    -149.483     144.751
Percent_Doctorate_degree                                                                 -58.9491    114.476     -0.515      0.607    -283.407     165.509
Percent_GED_or_alternative_credential                                                      0.6000     75.038      0.

In [ ]:
for l,u in zip(all_results_df_percent['CI_Low'], all_results_df_percent['CI_High']):
    if l <= 0 <= u:
        print("Not signficant")

In [ ]:
all_results_df_percent['Coeff Signficance'] = ['Not Signficant' if l <= 0 <= u else 'Signficant' for l,u in zip(all_results_df_percent['CI_Low'], all_results_df_percent['CI_High'])]

In [ ]:
all_results_df_percent

,Predictors,Coeff,Correlation,CI_Low,CI_High,Coeff Signficance
0,Percent_10th_grade,-51.876026,Negative,-207.448090,103.696037,Not Signficant
1,Percent_11th_grade,-25.266434,Negative,-194.126648,143.593780,Not Signficant
2,"Percent_12th_grade,_no_diploma",18.793564,Positive,-136.828351,174.415480,Not Signficant
3,Percent_1st_grade,402.856807,Positive,-271.517144,1077.230759,Not Signficant
4,Percent_2nd_grade,134.656530,Positive,-346.413840,615.726902,Not Signficant
...,...,...,...,...,...,...
61,Percent_75_79,41.489372,Positive,-417.149404,500.128167,Not Signficant
62,Percent_80_84,82.357470,Positive,-375.710004,540.424964,Not Signficant
63,Percent_85_over,140.640087,Positive,-323.894770,605.174964,Not Signficant
64,Estimate_Total_Income_in_the_past_12_months_at...,-0.000172,Negative,-0.000290,-0.000055,Signficant


In [ ]:
all_results_df_percent[all_results_df_percent['Coeff Signficance'] ==  'Signficant']

,Predictors,Coeff,Correlation,CI_Low,CI_High,Coeff Signficance
24,Percent_American_Indian_and_Alaska_Native_alone,119.334798,Positive,7.028707,237.344944,Signficant
25,Percent_Asian_alone,55.106160,Positive,6.013821,109.902554,Signficant
64,Estimate_Total_Income_in_the_past_12_months_at...,-0.000172,Negative,-0.000290,-0.000055,Signficant
65,Estimate_Total_Income_in_the_past_12_months_be...,0.000067,Positive,0.000008,0.000125,Signficant


In [ ]:
### Lasso Regression

In [ ]:
#lasso = Lasso(alpha=1.0)
#lasso.fit(x, y)

#lasso.coef_

array([ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        , -0.00025773,
        0.00012886])

In [ ]:
#pd.DataFrame({'Predictors': x.columns, 'Coeff': lasso.coef_})

,Predictors,Coeff
0,Percent_10th_grade,0.000000
1,Percent_11th_grade,0.000000
2,"Percent_12th_grade,_no_diploma",0.000000
3,Percent_1st_grade,0.000000
4,Percent_2nd_grade,0.000000
...,...,...
61,Percent_75_79,0.000000
62,Percent_80_84,0.000000
63,Percent_85_over,0.000000
64,Estimate_Total_Income_in_the_past_12_months_at...,-0.000258


In [ ]:
#### Ridge regression
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(x, y)
ridge_model.coef_

array([-3.44273396e+00,  2.65419841e+00,  3.16730251e+00,  1.44481311e-01,
        2.37105964e-02,  1.27716908e-01, -1.41771476e+00, -1.23516329e+00,
       -2.11407260e+00,  1.06238649e-01, -2.81959722e+00, -5.24173812e+00,
        3.30859143e+00,  6.23341533e+00, -1.08067951e+00,  1.28440698e+01,
       -2.14940085e-01, -9.09943446e+00,  1.13610109e+00, -2.73598945e+00,
       -5.74953868e+00,  4.19052204e+00,  2.13955954e+00,  9.58472768e-01,
        1.92852631e+00,  2.00016781e+00,  1.08846333e+00, -9.00939547e-01,
        3.88348702e+00,  3.41947884e+00,  3.38469925e+00,  3.47795920e-02,
       -1.14191838e+01, -4.77479647e+00,  2.96231328e+00, -7.38754517e+00,
       -3.30317586e+00,  3.68569108e+00,  6.18738901e-01, -9.05614734e-01,
       -4.45203980e-01,  4.77479647e+00, -2.00478435e+00, -4.44611982e+00,
       -5.51934968e+00, -3.03020209e+00,  3.20829174e+00, -8.85342266e-01,
        1.12957358e+00,  1.37237461e+00,  5.47785453e-01, -6.29823307e+00,
        7.79867036e-01, -

In [ ]:
pd.DataFrame({'Predictors': x.columns, 'Coeff': ridge_model.coef_})

,Predictors,Coeff
0,Percent_10th_grade,-3.442734
1,Percent_11th_grade,2.654198
2,"Percent_12th_grade,_no_diploma",3.167303
3,Percent_1st_grade,0.144481
4,Percent_2nd_grade,0.023711
...,...,...
61,Percent_75_79,0.369498
62,Percent_80_84,2.550236
63,Percent_85_over,4.683610
64,Estimate_Total_Income_in_the_past_12_months_at...,-0.000192


In [52]:
### Bootstrapping for coefficent CIs
nResample = 1000
ci = 0.95
coef_list= []

for i in range(nResample):
    x_boot, y_boot = resample(x, y)

    model_ridge_boot = Ridge(alpha=1.0).fit(x_boot, y_boot)

    #run_coefs = np.append(model_ridge_boot.coef_)
    coef_list.append(model_ridge_boot.coef_)


In [53]:
coef_df = pd.DataFrame(coef_list, columns=x.columns)

In [54]:
coef_df

,Percent_10th_grade,Percent_11th_grade,"Percent_12th_grade,_no_diploma",Percent_1st_grade,Percent_2nd_grade,Percent_3rd_grade,Percent_4th_grade,Percent_5th_grade,Percent_6th_grade,Percent_7th_grade,...,Percent_60_61,Percent_62_64,Percent_65_66,Percent_67_69,Percent_70_74,Percent_75_79,Percent_80_84,Percent_85_over,Estimate_Total_Income_in_the_past_12_months_at_or_above_the_poverty_level_Did_not_work,Estimate_Total_Income_in_the_past_12_months_below_the_poverty_level
0,-5.763218,2.599032,6.024047,0.766716,-0.373495,-0.907080,-1.322317,-1.429078,-3.162462,-2.578729,...,0.118669,5.484602,-0.166555,4.426953,4.872342,4.047002,6.076853,5.351409,-0.000236,0.000102
1,-2.744649,1.660373,0.720825,-0.069538,-0.056433,-2.134236,-1.212716,-0.582408,-3.699295,2.928426,...,0.451469,3.041112,-2.746099,3.052875,1.174940,0.363408,3.329659,4.308836,-0.000205,0.000093
2,-2.951279,4.415543,3.289169,-0.031782,0.117063,-1.602122,-1.660753,-0.024646,-2.812165,1.726324,...,2.031023,6.332359,-2.930400,-3.501401,-0.202520,0.689490,-0.605539,2.242064,-0.000138,0.000058
3,-6.066611,-0.692229,2.168771,0.472445,-0.119027,2.456780,-0.244753,-2.026147,-0.045526,-3.451217,...,2.652028,5.616731,0.966936,1.940297,7.971480,-0.810412,7.024025,6.874503,-0.000201,0.000095
4,-4.076133,2.305198,4.058096,-0.333232,-0.151137,-1.162706,-1.810700,-1.504401,-2.642574,2.456807,...,3.140074,3.689301,-6.727928,3.079748,0.777202,2.105095,2.056146,6.176312,-0.000115,0.000048
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1.257670,3.227282,2.384011,0.616952,0.283747,-0.336728,-1.120401,-1.036655,-2.294462,3.096704,...,4.492170,2.798238,-0.654277,1.488773,-0.916326,1.945601,3.004248,5.452269,-0.000067,0.000021
996,-4.255127,2.409567,-0.152052,-0.192599,0.466170,0.850941,-1.234991,-0.888532,0.279754,-1.569721,...,1.103529,6.121928,-1.244369,4.692261,4.304885,2.985618,5.704135,5.846810,-0.000140,0.000060
997,-2.689223,3.789333,1.347126,-0.210040,-0.015384,0.445246,-2.247448,-1.054578,-2.781089,-4.105388,...,1.279112,5.043303,0.164738,1.548771,2.834996,1.901035,2.710267,1.758321,-0.000133,0.000058
998,-1.652892,1.871586,2.571192,-0.027963,-0.037283,-0.942852,-1.043787,-1.013648,-1.763423,3.569243,...,4.137746,5.432468,-0.509427,4.027917,3.497023,1.453826,2.958580,5.713613,-0.000228,0.000103


In [55]:
lower_bound = (1.0 - ci) / 2*100
upper_bound = (1.0 + ci) / 2*100

ci_lower = np.percentile(coef_df, 2.5, axis=0)
ci_upper = np.percentile(coef_df, 97.5, axis=0)

In [59]:
coeff_mean = np.mean(coef_df, axis=0)
coeff_mean

Percent_10th_grade                                                                       -3.234521
Percent_11th_grade                                                                        2.657267
Percent_12th_grade,_no_diploma                                                            3.120084
Percent_1st_grade                                                                         0.134524
Percent_2nd_grade                                                                         0.020364
                                                                                            ...   
Percent_75_79                                                                             0.404708
Percent_80_84                                                                             2.418881
Percent_85_over                                                                           4.560287
Estimate_Total_Income_in_the_past_12_months_at_or_above_the_poverty_level_Did_not_work   -0.000191
Estimate_T

In [63]:
coeff_mean.to_numpy()

array([-3.23452135e+00,  2.65726690e+00,  3.12008418e+00,  1.34523806e-01,
        2.03637224e-02,  9.75611824e-02, -1.43893985e+00, -1.20356938e+00,
       -2.04188200e+00,  2.47399777e-01, -2.70432098e+00, -5.07984106e+00,
        3.02069478e+00,  6.04213839e+00, -1.04567584e+00,  1.27882178e+01,
       -2.13165523e-01, -9.14316347e+00,  1.25359075e+00, -2.55528125e+00,
       -5.72294421e+00,  4.09038914e+00,  2.05191483e+00,  1.01717947e+00,
        1.92322544e+00,  2.19001158e+00,  1.10020149e+00, -9.13260331e-01,
        3.78941684e+00,  3.33672102e+00,  3.22206217e+00,  1.14658857e-01,
       -1.14263160e+01, -4.81049298e+00,  2.81109619e+00, -7.22587935e+00,
       -3.22463750e+00,  3.64793999e+00,  5.90172638e-01, -9.36801907e-01,
       -4.72383050e-01,  4.81049298e+00, -1.91804310e+00, -4.39301460e+00,
       -5.44819217e+00, -2.93997769e+00,  3.10409182e+00, -8.62891565e-01,
        1.06012616e+00,  1.34698825e+00,  4.14006846e-01, -6.18169727e+00,
        6.61690044e-01, -

In [64]:
ridge_results_df_100 = pd.DataFrame({'Predictor': x.columns,
                                     'Mean': coeff_mean.to_numpy(),
                                'CI_Lower': ci_lower,
                                'CI_Upper': ci_upper})
ridge_results_df_100['Coeff Signficance'] = ['Not Signficant' if l <= 0 <= u else 'Signficant' for l,u in zip(ridge_results_df_100['CI_Lower'], ridge_results_df_100['CI_Upper'])]
#ridge_results_df.head()

In [65]:
ridge_results_df_100

,Predictor,Mean,CI_Lower,CI_Upper,Coeff Signficance
0,Percent_10th_grade,-3.234521,-7.624902,0.836919,Not Signficant
1,Percent_11th_grade,2.657267,-1.196601,6.298959,Not Signficant
2,"Percent_12th_grade,_no_diploma",3.120084,-0.473996,6.960203,Not Signficant
3,Percent_1st_grade,0.134524,-0.683781,0.703644,Not Signficant
4,Percent_2nd_grade,0.020364,-0.563153,0.609843,Not Signficant
...,...,...,...,...,...
61,Percent_75_79,0.404708,-4.216908,4.578989,Not Signficant
62,Percent_80_84,2.418881,-1.785190,6.454753,Not Signficant
63,Percent_85_over,4.560287,1.025897,8.084951,Signficant
64,Estimate_Total_Income_in_the_past_12_months_at...,-0.000191,-0.000319,-0.000065,Signficant


In [66]:
ridge_results_df_100[ridge_results_df_100['Coeff Signficance'] ==  'Signficant']

,Predictor,Mean,CI_Lower,CI_Upper,Coeff Signficance
15,Percent_GED_or_alternative_credential,12.788218,5.523160,20.878519,Signficant
17,Percent_Master's_degree,-9.143163,-16.337463,-1.901022,Signficant
21,Percent_Regular_high_school_diploma,4.090389,0.461861,7.795079,Signficant
32,Percent_White_alone,-11.426316,-15.772120,-7.137526,Signficant
33,Percent_Enrolled_in_school,-4.810493,-8.621672,-1.020287,Signficant
35,Percent_Enrolled_in_school_Enrolled_in_grade_1...,-7.225879,-14.573632,-1.071564,Signficant
41,Percent_Not_enrolled_in_school,4.810493,1.020287,8.621672,Signficant
44,Percent_10_14,-5.448192,-10.204081,-0.999139,Signficant
51,Percent_35_39,-6.181697,-12.424733,-0.108662,Signficant
57,Percent_62_64,6.350203,2.360821,10.121596,Signficant
